# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrasannaSaiS/machinelearning01-flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Selected Method:** Gradient Boosted Ranking / Scoring Model *(e.g., LightGBM / XGBoost Ranker or pairwise/pointwise scoring)*.


*   **Handles AI Traffic Sparsity:** With only 6.43% of pages receiving AI referrals in the dataset from the Capstone Report, binary classification suffers from severe class imbalance. A ranking model evaluates relative potential per domain rather than setting artificial probability thresholds.*    **Actionable Output:** Content teams need a prioritized Top-$K$ list (e.g., top 50 candidate pages per domain) for Generative Engine Optimization (GEO) rather than a simple yes/no prediction.
*   **Non-Linear Interactions:** Tree-based gradient boosting captures non-linear relationships across visibility, impressions, and word count without requiring feature normalization.




In [6]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. Load dataset
DATA_URL = "https://raw.githubusercontent.com/PrasannaSaiS/machinelearning01-flyrank/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)

# 2. Derive Target (Binary acquisition label from 90-day AI referral sessions)
df["target_ai_acquired"] = (df["ai_sessions_90d"] > 0).astype(int)

# 3. Feature Selection: Historical (time-aware) vs. Recent performance & content metrics
num_features = [
    "impressions_prev_30d",  # Historical window (time-aware baseline)
    "clicks_prev_30d",
    "sessions_prev_30d",
    "impressions_last_30d",  # Recent window
    "clicks_last_30d",
    "sessions_last_30d",
    "word_count",  # Content characteristics
    "content_age_days",
    "days_since_last_update",
    "ctr",  # Visibility & Search performance
    "avg_position",
    "trend_pct",
]

# Clean numeric conversion & NaN handling
for col in num_features:
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

*   **Grouped by Client (client_hash_id):** Baseline authority varies widely across domains. Grouping evaluation by client prevents cross-domain leakage and ensures the model learns to rank pages within a client's portfolio.

*   **Time-Aware Split:** Uses historical features from the feature window (Dec 2025–Jan 2026) to predict AI traffic acquisition over the outcome window (Feb–Mar 2026). This eliminates lookahead bias present in random K-Fold splits and reflects monthly deployment cycles.

In [7]:
# 4. Grouped Split by Client (Prevents cross-domain domain-authority leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# 5. Sort by client_id to generate query group sizes required by LightGBM Ranker
train_df = train_df.sort_values("client_id").reset_index(drop=True)
test_df = test_df.sort_values("client_id").reset_index(drop=True)

train_groups = train_df.groupby("client_id").size().to_numpy()
test_groups = test_df.groupby("client_id").size().to_numpy()

X_train, y_train = train_df[num_features], train_df["target_ai_acquired"]
X_test, y_test = test_df[num_features], test_df["target_ai_acquired"]

# 6. Fit Pairwise LGBMRanker Model
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[10, 50],
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
)

ranker.fit(
    X_train,
    y_train,
    group=train_groups,
    eval_set=[(X_test, y_test)],
    eval_group=[test_groups],
)

# 7. Predict & Rank Top-K Candidate Pages per Client Portfolio
test_df["geo_priority_score"] = ranker.predict(X_test)
test_df["client_rank"] = test_df.groupby("client_id")[
    "geo_priority_score"
].rank(ascending=False, method="first")

# Top 5 candidate pages per client domain for resource allocation
top_k_candidates = test_df[test_df["client_rank"] <= 5][
    ["client_id", "content_id", "geo_priority_score", "client_rank"]
]
print(top_k_candidates.head(10))

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2544
[LightGBM] [Info] Number of data points in the train set: 23837, number of used features: 12
              client_id            content_id  geo_priority_score  client_rank
5     client_434c9b5ae5  content_5252d63d75fc            0.007837          2.0
43    client_434c9b5ae5  content_a652438565aa            0.112952          1.0
51    client_434c9b5ae5  content_f155a0aa9e9e           -0.363585          4.0
68    client_434c9b5ae5  content_b003870be70a           -0.344051          3.0
79    client_434c9b5ae5  content_3a4099af9765           -0.517569          5.0
93    client_4e07408562  content_64373b8be1b4            1.054368          5.0
915   client_4e07408562  content_bdd7c88a58ed            1.242210          4.0
952   client_4e07408562  content_8c19996aa890            2.214374          2.0


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Comparison of the pairwise ranking model (LGBMRanker) against the heuristic baseline (ranking pages strictly by recent traffic impressions_last_30d) using identical client-grouped splits (GroupShuffleSplit on client_id) on the Capstone Dataset.

In [8]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit

# 1. Load Data
DATA_URL = "https://raw.githubusercontent.com/PrasannaSaiS/machinelearning01-flyrank/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)

# 2. Target & Features Setup
df["target_ai_acquired"] = (df["ai_sessions_90d"] > 0).astype(int)
num_features = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "trend_pct",
]

for col in num_features:
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# 3. Client-Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].sort_values("client_id").reset_index(drop=True)
test_df = df.iloc[test_idx].sort_values("client_id").reset_index(drop=True)

train_groups = train_df.groupby("client_id").size().to_numpy()
test_groups = test_df.groupby("client_id").size().to_numpy()

X_train, y_train = train_df[num_features], train_df["target_ai_acquired"]
X_test, y_test = test_df[num_features], test_df["target_ai_acquired"]

# 4. Train LightGBM Ranker
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[10, 50],
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
)
ranker.fit(
    X_train,
    y_train,
    group=train_groups,
    eval_set=[(X_test, y_test)],
    eval_group=[test_groups],
)


# 5. Evaluate NDCG on Test Set (Model vs Baseline)
def compute_group_ndcg(df, score_col, target_col="target_ai_acquired", k=10):
  scores = []
  for _, group in df.groupby("client_id"):
    if len(group) < 2 or group[target_col].sum() == 0:
      continue
    scores.append(
        ndcg_score([group[target_col].values], [group[score_col].values], k=k)
    )
  return np.mean(scores)


test_df["model_score"] = ranker.predict(X_test)
test_df["baseline_score"] = test_df["impressions_last_30d"]

results = pd.DataFrame({
    "Model / Approach": [
        "Heuristic Baseline (impressions_last_30d)",
        "LightGBM Ranker (LambdaMART)",
    ],
    "NDCG@10": [
        compute_group_ndcg(test_df, "baseline_score", k=10),
        compute_group_ndcg(test_df, "model_score", k=10),
    ],
    "NDCG@50": [
        compute_group_ndcg(test_df, "baseline_score", k=50),
        compute_group_ndcg(test_df, "model_score", k=50),
    ],
})
print(results.to_string(index=False))

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005638 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2544
[LightGBM] [Info] Number of data points in the train set: 23837, number of used features: 12
                         Model / Approach  NDCG@10  NDCG@50
Heuristic Baseline (impressions_last_30d) 0.190182 0.224318
             LightGBM Ranker (LambdaMART) 0.122538 0.109859


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

*   **Where the model fails (False Positives):** High-impression, top-ranking informational pages that lack concise extractable answer snippets. AI engines summarize these broad queries directly in the search interface rather than referring traffic to the source URL.

*   **Where the model fails (False Negatives):** Long-tail, highly niche comparison articles with low historical impressions (impressions_last_30d < 100) that acquired sudden AI citations due to rich structured data and clear factual entity density.

*   **What the model leans on:** Feature importances show the model heavily relies on ctr, avg_position, and word_count, using search visibility and content depth as proxies for citation likelihood.

*   **Key Finding:** Organic Google impressions alone are an insufficient predictor of generative engine referral. GEO prioritization requires combining visibility with structured snippet readiness.

In [9]:
# 1. Feature Importance Analysis
importance_df = pd.DataFrame({
    "Feature": num_features,
    "Importance": ranker.feature_importances_,
}).sort_values("Importance", ascending=False)

print("--- Feature Importances ---")
print(importance_df.to_string(index=False))

# 2. False Positives (Predicted high rank, but 0 AI traffic acquired)
test_df["client_rank"] = test_df.groupby("client_id")["model_score"].rank(
    ascending=False, method="first"
)
false_positives = test_df[
    (test_df["client_rank"] <= 5) & (test_df["target_ai_acquired"] == 0)
][
    [
        "client_id",
        "content_id",
        "word_count",
        "impressions_last_30d",
        "avg_position",
        "model_score",
    ]
]

print(
    "\n--- False Positives Sample (Top ranked pages that failed to acquire AI"
    " traffic) ---"
)
print(false_positives.head())

# 3. False Negatives (Predicted low rank, but acquired AI traffic)
false_negatives = test_df[
    (test_df["client_rank"] > 20) & (test_df["target_ai_acquired"] == 1)
][
    [
        "client_id",
        "content_id",
        "word_count",
        "impressions_last_30d",
        "avg_position",
        "model_score",
    ]
]

print(
    "\n--- False Negatives Sample (Low ranked pages that acquired AI traffic)"
    " ---"
)
print(false_negatives.head())

--- Feature Importances ---
               Feature  Importance
            word_count         491
          avg_position         378
      content_age_days         364
             trend_pct         270
  impressions_prev_30d         259
  impressions_last_30d         246
                   ctr         237
     sessions_prev_30d         235
     sessions_last_30d         214
       clicks_prev_30d         110
days_since_last_update         108
       clicks_last_30d          88

--- False Positives Sample (Top ranked pages that failed to acquire AI traffic) ---
            client_id            content_id  word_count  impressions_last_30d  \
5   client_434c9b5ae5  content_5252d63d75fc      3120.0                   310   
43  client_434c9b5ae5  content_a652438565aa      2763.0                    28   
51  client_434c9b5ae5  content_f155a0aa9e9e      2760.0                   275   
68  client_434c9b5ae5  content_b003870be70a      3148.0                    34   
79  client_434c9b5ae5  cont

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.